In [24]:
import pandas as pd
import numpy as np
import os
import pyreadstat

# Base

In [33]:
os.chdir("/Users/cristianmolina/Desktop/Proyectos MINEDU 2026 - Github/MINEDU-2026-05-INFOGRAFIA-ASCENSO-EB-MINEDU-MINDEF-2024/")

In [36]:
Base, meta = pyreadstat.read_sav("data/Consolidado_Ascenso-EB-2024_Minedu+Mindef_actualizado 1.sav")

In [37]:
meta.variable_value_labels

{'TIPODOC': {8.0: 'DNI', 9.0: 'C.EXT.'},
 'sex': {1.0: 'Masculino', 2.0: 'Femenino'},
 'edquin': {2.0: 'De 27 a 29 años',
  3.0: 'De 30 a 34 años',
  4.0: 'De 35 a 39 años',
  5.0: 'De 40 a 44 años',
  6.0: 'De 45 a 49 años',
  7.0: 'De 50 a 54 años',
  8.0: 'De 55 a 59 años',
  9.0: 'De 60 a 64 años'},
 'grupedad': {1.0: 'De 27 a 29 años',
  2.0: 'De 30 a 39 años',
  3.0: 'De 40 a 49 años',
  4.0: 'De 50 a 59 años',
  5.0: 'De 60 a más años'},
 'LEY_PROCEDE': {1.0: 'CPM', 2.0: 'PROF', 3.0: 'LRM'},
 'BASE': {1.0: 'MINEDU', 2.0: 'MINDEF'}}

In [38]:
print(Base["RENUNCIA"].value_counts(dropna=False))
print(Base[Base["RENUNCIA"]==1]["DOCUMENTO"])
print(Base[Base["DOCUMENTO"].duplicated(keep=False)]["DOCUMENTO"])

RENUNCIA
0.0    89932
1.0        4
Name: count, dtype: int64
33605    02437852
44744    08941812
64904    23848050
79655    31659478
Name: DOCUMENTO, dtype: object
79655    31659478
79656    31659478
Name: DOCUMENTO, dtype: object


In [39]:
Base[Base["RENUNCIA"]!=1]["BASE"].value_counts(dropna=False)

BASE
1.0    89754
2.0      178
Name: count, dtype: int64

# Tablas

### T1. Participación docente por etapas 

In [47]:
BD = Base[Base["RENUNCIA"]!=1].copy()

In [49]:
orden_escalas = [
    "18-24",
    "25-29",
    "30-34",
    "35-39",
    "40-44",
    "45-49",
    "50-54",
    "55-59",
    "60 a más"
]

BD["RANGO_EDAD_IDR"] = np.select(
       [ (BD["EDAD"] >= 18) & (BD["EDAD"] <= 24),
        (BD["EDAD"] >= 25) & (BD["EDAD"] <= 29),
        (BD["EDAD"] >= 30) & (BD["EDAD"] <= 34),
        (BD["EDAD"] >= 35) & (BD["EDAD"] <= 39),
        (BD["EDAD"] >= 40) & (BD["EDAD"] <= 44),
        (BD["EDAD"] >= 45) & (BD["EDAD"] <= 49),
        (BD["EDAD"] >= 50) & (BD["EDAD"] <= 54),
        (BD["EDAD"] >= 55) & (BD["EDAD"] <= 59),
        (BD["EDAD"] >= 60)],
    [
        "18-24",
        "25-29",
        "30-34",
        "35-39",
        "40-44",
        "45-49",
        "50-54",
        "55-59",
        "60 a más"
    ],  
    default = "-"
)




In [41]:
resumen = pd.DataFrame({
    "Inscritos": [
        BD[BD['BASE'] == 1].shape[0],
        BD[BD['BASE'] == 2].shape[0]
    ],
    "Evaluados Etapa Nac.": [
        BD[
            (BD['BASE'] == 1) &
            (BD['ESTADO_PUN'].isin(['SUPERÓ PUNTAJE MÍNIMO', 'NO SUPERÓ PUNTAJE MÍNIMO']))
        ].shape[0],
        BD[
            (BD['BASE'] == 2) &
            (BD['ESTADO_PUN'].isin(['SUPERÓ PUNTAJE MÍNIMO', 'NO SUPERÓ PUNTAJE MÍNIMO']))
        ].shape[0]
    ],
    "Evaluados Etapa Des.": [
        BD[
            (BD['BASE'] == 1) &
            (BD['EVAL_ED'].isin([1]))
        ].shape[0],
        BD[
            (BD['BASE'] == 2) &
            (BD['EVAL_ED'].isin([1]))
        ].shape[0]
    ],
    "Ganadores": [
        BD[(BD['BASE'] == 1) & (BD['GANADOR_ASCENSO'] == 1)].shape[0],
        BD[(BD['BASE'] == 2) & (BD['GANADOR_ASCENSO'] == 1)].shape[0]
    ]
}, 
index=['Minedu', 'Mindef'])

# Fila Total
resumen.loc['Total'] = resumen.sum()

resumen.reset_index(inplace=True)
T1 = resumen.copy()

T1.to_excel(r"output/T1.xlsx", 
            index=False)

### T2. Evaluados y aprobados de la Etapa Nacional, y ganadores por escala magisterial 

In [42]:
orden_escalas = [
    "Segunda Escala",
    "Tercera Escala",
    "Cuarta Escala",
    "Quinta Escala",
    "Sexta Escala",
    "Séptima Escala",
    "Octava Escala"
]

def conteo(mask, nombre):
    return (
        BD.loc[mask, "ESCALA_POSTULA_DESCRIPCION"]
        .value_counts()
        .reindex(orden_escalas, fill_value=0)
        .rename(nombre)
    )

tabla = pd.concat([
    conteo(
        BD["ESTADO_PUN"].isin([
            "SUPERÓ PUNTAJE MÍNIMO",
            "NO SUPERÓ PUNTAJE MÍNIMO"
        ]),
        "Evaluados Etapa Nac."
    ),
    conteo(
        BD["ESTADO_PUN"].eq("SUPERÓ PUNTAJE MÍNIMO"),
        "Aprobados Etapa Nac."
    ),
    conteo(
        BD["GANADOR_ASCENSO"].eq(1),
        "Ganadores"
    )
], axis=1)

tabla.loc["Total"] = tabla.sum()
tabla.reset_index(inplace=True)
T2 = tabla.copy()
T2["Ganadores/Evaluados"] = round(T2["Ganadores"] / T2["Evaluados Etapa Nac."] * 100, 0)
T2
T2.to_excel("output/T2.xlsx", 
            index=False)

### T3. Ganadores por sexo

In [43]:
resultado = (
    BD.query("GANADOR_ASCENSO == 1")
    .groupby(["BASE", "SEXO_POSTULANTE"])
    .size()
    .unstack(fill_value=0)
    .assign(
        Total=lambda x: x.sum(axis=1),
        Mujeres_pct=lambda x: round(x["FEMENINO"] / x["Total"]*100,0),
        Varones_pct=lambda x: round(x["MASCULINO"] / x["Total"]*100,0)
    )
    .pipe(
        lambda x: pd.concat([
            x,
            pd.DataFrame({
                "FEMENINO": [x["FEMENINO"].sum()],
                "MASCULINO": [x["MASCULINO"].sum()],
                "Total": [x["Total"].sum()],
                "Mujeres_pct": [round(x["FEMENINO"].sum() / x["Total"].sum()*100,0)],
                "Varones_pct": [round(x["MASCULINO"].sum() / x["Total"].sum()*100,0)]
            }, index=["Total"])
        ])
    )
).reset_index()

T3 = resultado.copy()
T3

T3.to_excel("output/T3.xlsx", 
            index=False)

### T4. Ganadores segun rango de edad

In [50]:
orden_escalas = [
    "18-24",
    "25-29",
    "30-34",
    "35-39",
    "40-44",
    "45-49",
    "50-54",
    "55-59",
    "60 a más"
]

def conteo(mask, nombre):
    return (
        BD.loc[mask, "RANGO_EDAD_IDR"]
        .value_counts()
        .reindex(orden_escalas, fill_value=0)
        .rename(nombre)
    )

tabla = (
    pd.concat([
        conteo(
            BD["ESTADO_PUN"].isin([
                "SUPERÓ PUNTAJE MÍNIMO",
                "NO SUPERÓ PUNTAJE MÍNIMO"
            ]),
            "Evaluados Etapa Nac."
        ),
        conteo(
            BD["GANADOR_ASCENSO"].eq(1),
            "Ganadores"
        )
    ], axis=1)
    .assign(
        **{
            "%_Ganadores": lambda x: round(
                x["Ganadores"] / x["Ganadores"].sum() * 100,
                0
            )
        }
    )
)

tabla.loc["Total"] = [
    tabla["Evaluados Etapa Nac."].sum(),
    tabla["Ganadores"].sum(),
    tabla["%_Ganadores"].sum()  # Esto sumará los porcentajes, lo cual no es correcto, pero se deja para mantener la estructura
]

tabla.reset_index(inplace=True)
tabla.rename(columns={"index": "RANGO_EDAD_IDR"}, inplace=True)

T4 = tabla.copy()
T4["Ganadores_Evaluados"] = round(T4["Ganadores"] / T4["Evaluados Etapa Nac."] * 100, 0)
T4

T4.to_excel("output/T4.xlsx", 
            index=False)

### T5. Ganadores según discapacidad

In [63]:
BD["BONIFICACION_DISCAPACIDAD_A"] = pd.to_numeric(BD["BONIFICACION_DISCAPACIDAD_A"], errors='coerce')

BD["RECIBIO_BONIF_DISPAC_IDR"] = np.where(
    BD["BONIFICACION_DISCAPACIDAD_A"] != 0,
    "Sí",
    "No"
)

In [64]:
T5 = pd.concat([
    BD.loc[
        BD["ESTADO_PUN"].isin([
            "SUPERÓ PUNTAJE MÍNIMO",
            "NO SUPERÓ PUNTAJE MÍNIMO"
        ]),
        "TIENE_DISCAPACIDAD"
    ]
    .value_counts()
    .rename("Evaluados"),

    BD.loc[
        BD["GANADOR_ASCENSO"].eq(1),
        "RECIBIO_BONIF_DISPAC_IDR"
    ]
    .value_counts()
    .rename("Ganadores")
], axis=1)\
        .pipe(
            lambda x: x.assign(**{"Ganadores_%": round(x["Ganadores"] / x["Ganadores"].sum() * 100, 0)})
        )

T5.index = ["Sin discapacidad", "Con discapacidad"]

T5.loc["Total"] = T5.sum()
T5["Ganadores_sobre_Evaluados"] = round(T5["Ganadores"] / T5["Evaluados"] * 100, 0)

T5.reset_index(inplace=True)
T5.rename(columns={"index": "Condición de discapacidad"}, inplace=True)

T5

T5.to_excel("output/T5.xlsx", 
            index=False)

### T6_A y T6_B

In [68]:
tabla = (
    BD.groupby(["BASE", "REGION_LEGAJO_ACTUALIZADO"])
    .agg(
        Evaluados_PUN=("ESTADO_PUN",
                       lambda x: x.isin([
                           "SUPERÓ PUNTAJE MÍNIMO",
                           "NO SUPERÓ PUNTAJE MÍNIMO"
                       ]).sum()),
        Aprobados_PUN=("ESTADO_PUN",
                       lambda x: x.eq("SUPERÓ PUNTAJE MÍNIMO").sum()),
        Reprobados_PUN=("ESTADO_PUN",
                        lambda x: x.eq("NO SUPERÓ PUNTAJE MÍNIMO").sum())
    )
    .reset_index()
    .assign(
        Aprobados_PUN_pct=lambda x: round(
            x["Aprobados_PUN"] / x["Evaluados_PUN"] * 100,
            0
        )
    )
)

tabla.loc["Total"] = [
    "",
    "TOTAL",
    tabla["Evaluados_PUN"].sum(),
    tabla["Aprobados_PUN"].sum(),
    tabla["Reprobados_PUN"].sum(),
    round(
        tabla["Aprobados_PUN"].sum()
        / tabla["Evaluados_PUN"].sum()
        * 100,
        0
    )
]

tabla

tabla.to_excel("output/T6_A.xlsx", 
            index=False)

In [72]:
BD.columns.to_list()

['ID',
 'TIPODOC',
 'DOCUMENTO',
 'RENUNCIA',
 'APEPAT',
 'APEMAT',
 'NOMBRES',
 'SEXO_POSTULANTE',
 'sex',
 'FECHA_NACIMIENTO',
 'EDAD',
 'RANGO_EDAD',
 'edquin',
 'grupedad',
 'CORREO',
 'CELULAR',
 'ID_LENGUA',
 'LENGUA_MATERNA',
 'TIENE_DISCAPACIDAD',
 'DISCAPACIDAD',
 'TIPO_DISCAPACIDAD',
 'REQ_APOYO_DISCAPCIDAD',
 'ID_MODALIDAD',
 'MODALIDAD',
 'ID_NIVEL_CICLO',
 'NIVEL_CICLO',
 'ID_ESPECALIDAD',
 'ESPECALIDAD',
 'GRUPO_INSCRIPCION_FINAL',
 'ESCALA_ACTUAL_NUMERO',
 'ESCALA_ACTUAL_DESCRIPCION',
 'ESCALA_POSTULA_NUMERO',
 'ESCALA_POSTULA_DESCRIPCION',
 'REGION_LEGAJO_ACTUALIZADO',
 'DRE_UGEL_LEGAJO_ACTUALIZADO',
 'ID_SEDE_PN_INSCRIPCION',
 'SEDE_PN_INSCRIPCION',
 'CUMPLIMIENTO_REQUISITO_1',
 'CUMPLIMIENTO_REQUISITO_2',
 'PROCEDE_DE_INSTITUTO',
 'NOMBRE_DE_INSTITUTO',
 'NOMBRE_OTRO_INSTITUTO',
 'PROCEDE_DE_UNIVERSIDAD',
 'NOMBRE_DE_UNIVERSIDAD',
 'NOMBRE_OTRA_UNIVERSIDAD',
 'MEDIO_COMUNICACION',
 'AUTORIZACION_DATOS_PERSONAL',
 'INFO_VERAZ',
 'CODIGO_VALIDACION',
 'REGION_EVALUACION

In [ ]:
tabla = (
    BD.groupby(["BASE", "REGION_LEGAJO_ACTUALIZADO"])
    .agg(
        Evaluados_PUN=("ESTADO_PUN",
                       lambda x: x.isin([
                           "SUPERÓ PUNTAJE MÍNIMO",
                           "NO SUPERÓ PUNTAJE MÍNIMO"
                       ]).sum()),
        Aprobados_PUN=("ESTADO_PUN",
                       lambda x: x.eq("SUPERÓ PUNTAJE MÍNIMO").sum()),
        Reprobados_PUN=("ESTADO_PUN",
                        lambda x: x.eq("NO SUPERÓ PUNTAJE MÍNIMO").sum())
    )
    .reset_index()
    .assign(
        Aprobados_PUN_pct=lambda x: round(
            x["Aprobados_PUN"] / x["Evaluados_PUN"] * 100,
            0
        )
    )
)

tabla.loc["Total"] = [
    "",
    "TOTAL",
    tabla["Evaluados_PUN"].sum(),
    tabla["Aprobados_PUN"].sum(),
    tabla["Reprobados_PUN"].sum(),
    round(
        tabla["Aprobados_PUN"].sum()
        / tabla["Evaluados_PUN"].sum()
        * 100,
        0
    )
]

tabla_2 = (
    BD.groupby(["BASE", "DESC_UNIDAD_EJECUTORA"])
    .agg(
        Evaluados_PUN=("ESTADO_PUN",
                       lambda x: x.isin([
                           "SUPERÓ PUNTAJE MÍNIMO",
                           "NO SUPERÓ PUNTAJE MÍNIMO"
                       ]).sum()),
        Aprobados_PUN=("ESTADO_PUN",
                       lambda x: x.eq("SUPERÓ PUNTAJE MÍNIMO").sum()),
        Reprobados_PUN=("ESTADO_PUN",
                        lambda x: x.eq("NO SUPERÓ PUNTAJE MÍNIMO").sum())
    )
    .reset_index()
    .assign(
        Aprobados_PUN_pct=lambda x: round(
            x["Aprobados_PUN"] / x["Evaluados_PUN"] * 100,
            0
        )
    )
)

tabla_2.loc["Total"] = [
    "",
    "TOTAL",
    tabla_2["Evaluados_PUN"].sum(),
    tabla_2["Aprobados_PUN"].sum(),
    tabla_2["Reprobados_PUN"].sum(),
    round(
        tabla_2["Aprobados_PUN"].sum()
        / tabla_2["Evaluados_PUN"].sum()
        * 100,
        0
    )
]

tabla_2

tabla_2.to_excel("output/T6_B.xlsx", 
            index=False)

### T7

In [74]:
tabla = (
    BD.groupby(["GRUPO_INSCRIPCION_FINAL"])
    .agg(
        Evaluados_PUN=("ESTADO_PUN",
                       lambda x: x.isin([
                           "SUPERÓ PUNTAJE MÍNIMO",
                           "NO SUPERÓ PUNTAJE MÍNIMO"
                       ]).sum()),
        Aprobados_PUN=("ESTADO_PUN",
                       lambda x: x.eq("SUPERÓ PUNTAJE MÍNIMO").sum()),
        Reprobados_PUN=("ESTADO_PUN",
                        lambda x: x.eq("NO SUPERÓ PUNTAJE MÍNIMO").sum())
    )
    .reset_index()
    .assign(
        Aprobados_PUN_pct=lambda x: round(
            x["Aprobados_PUN"] / x["Evaluados_PUN"] * 100,
            0
        )
    )
)

tabla.loc["Total"] = [
    "TOTAL",
    tabla["Evaluados_PUN"].sum(),
    tabla["Aprobados_PUN"].sum(),
    tabla["Reprobados_PUN"].sum(),
    round(
        tabla["Aprobados_PUN"].sum()
        / tabla["Evaluados_PUN"].sum()
        * 100,
        0
    )
]

tabla

tabla.to_excel("output/T7.xlsx", 
            index=False)

### T8

In [76]:
BD["TRAYECTORIA_1_1_MASTER"] = np.where(BD["TRAYECTORIA_1_1_GRADO"] == "Grado de Maestro/Maestría/Magíster/Máster", "Sí", "No")
BD["TRAYECTORIA_1_1_DOCTOR"] = np.where(BD["TRAYECTORIA_1_1_GRADO"] == "Grado de Doctor", "Sí", "No")

In [77]:
COLUMNAS_ED = [
    'TRAYECTORIA_1_1_MASTER',
    'TRAYECTORIA_1_1_DOCTOR',
    'TRAYECTORIA_1_2',
    'TRAYECTORIA_1_3',
    'TRAYECTORIA_1_4',
    'TRAYECTORIA_1_5',
    'TRAYECTORIA_2_1',
    'TRAYECTORIA_2_2',
    'TRAYECTORIA_2_3',
    'TRAYECTORIA_2_4',
    'TRAYECTORIA_2_5A',
    'TRAYECTORIA_2_5B',
    'TRAYECTORIA_2_5C',
    'TRAYECTORIA_2_5D',
    'TRAYECTORIA_3_1A',
    'TRAYECTORIA_3_1B',
    'TRAYECTORIA_3_1C',
    'TRAYECTORIA_3_2',
    'TRAYECTORIA_3_3',
    'TRAYECTORIA_3_4',
    'TRAYECTORIA_3_5',
    'TRAYECTORIA_3_6'
]

T8 = (
    BD.loc[
        BD["EVAL_ED"].isin([
            1
        ]),
        COLUMNAS_ED
    ]
    .eq("Sí")
    .sum()
    .reset_index(name="Cumplieron")
    .rename(columns={"index": "Criterio"})
    .assign(
        Evaluados=lambda x: BD["EVAL_ED"].isin([
            1]).sum(),
        Criterio=lambda x: (
            x["Criterio"]
            .str.replace("TRAYECTORIA_", "Criterio ", regex=False)
            .str.replace("_GRADO", "a", regex=False)
            .str.replace("_", ".", regex=False)
            .str.lower()
            .str.replace(".5a", ".5a")
            .str.replace(".5b", ".5b")
            .str.replace(".5c", ".5c")
            .str.replace(".5d", ".5d")
            .str.title()
        )
    )
    [["Criterio", "Evaluados", "Cumplieron"]]
)

T8["Cumplimiento_pct"] = round(T8["Cumplieron"] / T8["Evaluados"] * 100, 0)

T8.to_excel("output/T8.xlsx", 
            index=False)

In [82]:
BD[BD["EVAL_ED"]==1]["BASE"].value_counts(dropna=False)

BASE
1.0    24808
2.0       96
Name: count, dtype: int64

In [80]:
T8

,Criterio,Evaluados,Cumplieron,Cumplimiento_pct
0,Criterio 1.1.Master,24904,6254,25.0
1,Criterio 1.1.Doctor,24904,625,3.0
2,Criterio 1.2,24904,3781,15.0
3,Criterio 1.3,24904,12202,49.0
4,Criterio 1.4,24904,14779,59.0
5,Criterio 1.5,24904,17016,68.0
6,Criterio 2.1,24904,49,0.0
7,Criterio 2.2,24904,310,1.0
8,Criterio 2.3,24904,762,3.0
9,Criterio 2.4,24904,1159,5.0


# Adicional

In [79]:
BD[BD["BASE"]==2]["ESTADO_PUN"].value_counts(dropna=False)

ESTADO_PUN
SUPERÓ PUNTAJE MÍNIMO       96
NO SUPERÓ PUNTAJE MÍNIMO    80
NO ASISTIÓ                   2
Name: count, dtype: int64